<!-- chinese-cell-note -->
#### Cell 01 中文说明

**用途**：导入碳核算、模型复用和统计评估依赖，并建立 Step 4 输出目录。

**输入与输出**：输入为前序输出路径；输出为 OCEI 分析目录和基础配置。

**评委回应重点**：它将碳分析环境与 EUI 模型训练环境清晰分离。


<!-- english-cell-note -->
#### Cell 01 - Environment for carbon-coupling analysis

**Purpose**: This cell imports carbon-accounting, model-reuse, and evaluation utilities for Step 4.

**Inputs and outputs**: Defines deterministic paths, folder handles, plotting defaults, and configuration objects used by downstream cells.

**Reviewer-facing rationale**: Translates the EUI-OCEI divergence into ranked evidence that can be checked against the manuscript claims.


In [ ]:


from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.metrics import (
    mean_absolute_percentage_error,
    mean_squared_error,
    mean_absolute_error,
    r2_score
)
from sklearn.model_selection import KFold

plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.20,
    "grid.linestyle": "--",
})

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
STEP2_DIR = PROJECT_ROOT / "outputs_step2"
STEP3_DIR = PROJECT_ROOT / "outputs_step3"
OUT_DIR = PROJECT_ROOT / "outputs_step4"
FIG_DIR = OUT_DIR / "figures"
MODEL_DIR = OUT_DIR / "models"
for p in [OUT_DIR, FIG_DIR, MODEL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

DATASET_PATH = DATA_DIR / "step1_simulation_dataset.csv"
SRC_PATH = STEP2_DIR / "src_indices_bootstrap.csv"
BEST2_PATH = STEP3_DIR / "best2_models.csv"
METRICS_PATH = STEP3_DIR / "model_metrics.csv"

<!-- chinese-cell-note -->
#### Cell 02 中文说明

**用途**：定义电力、天然气、区域供热和区域供冷排放因子，并明确终端能耗到载体的映射。

**输入与输出**：输入为论文采用的排放因子；输出为 EMISSION_FACTORS 与载体映射。

**评委回应重点**：它回应排放因子来源、年份和核算边界必须明确的问题。


<!-- english-cell-note -->
#### Cell 02 - Carbon-accounting boundary

**Purpose**: This cell defines carrier-specific emission factors and maps end-use energy to carbon-accounting carriers.

**Inputs and outputs**: Uses the declared emission-factor assumptions to construct OCEI and export coupling evidence.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
# ---------- 1) Configure carbon-emission factors ----------
# Unit: kgCO2e / kWh. The baseline factors follow the manuscript accounting boundary.
# District-cooling EF is derived from electricity EF, a system COP, and auxiliary electricity.
DISTRICT_COOLING_COP_BASELINE = 4.5
DISTRICT_COOLING_AUXILIARY_EF = 0.038
DISTRICT_COOLING_EF_BASELINE = 0.55 / DISTRICT_COOLING_COP_BASELINE + DISTRICT_COOLING_AUXILIARY_EF

EMISSION_FACTORS = {
    "electricity": 0.55,
    "natural_gas": 0.202,
    "district_heating": 0.22,
    "district_cooling": DISTRICT_COOLING_EF_BASELINE,
}

emission_factor_sources = pd.DataFrame([
    {"carrier": "electricity", "baseline_factor_kgco2e_per_kwh": EMISSION_FACTORS["electricity"], "basis": "North China regional grid factor used in the manuscript accounting boundary", "uncertainty_treatment": "Low/high electricity and illustrative grid-decarbonisation scenarios"},
    {"carrier": "natural_gas", "baseline_factor_kgco2e_per_kwh": EMISSION_FACTORS["natural_gas"], "basis": "Natural-gas combustion factor used in the manuscript accounting boundary", "uncertainty_treatment": "Held constant in grid and COP stress tests"},
    {"carrier": "district_heating", "baseline_factor_kgco2e_per_kwh": EMISSION_FACTORS["district_heating"], "basis": "District-heating factor used in the manuscript accounting boundary", "uncertainty_treatment": "Held constant except in explicit heating stress tests if added"},
    {"carrier": "district_cooling", "baseline_factor_kgco2e_per_kwh": EMISSION_FACTORS["district_cooling"], "basis": "Derived as electricity_factor / COP + auxiliary_EF = 0.55 / 4.5 + 0.038", "uncertainty_treatment": "COP sensitivity scenarios with COP = 3.5 and COP = 5.5"},
])
emission_factor_sources.to_csv(OUT_DIR / "emission_factor_sources.csv", index=False, encoding="utf-8-sig")

# ---------- Fixed carbon-accounting boundary ----------
# 1) Space heating -> district_heating.
# 2) Space cooling -> district_cooling.
# 3) Domestic hot water -> natural_gas.
# 4) Building-side electricity loads such as lighting, equipment, and fans -> electricity.
SHOW_ZERO_CARRIERS = True

print("District-cooling EF derivation:")
print(f"0.55 / {DISTRICT_COOLING_COP_BASELINE:.1f} + {DISTRICT_COOLING_AUXILIARY_EF:.3f} = {DISTRICT_COOLING_EF_BASELINE:.3f} kgCO2e/kWh")
emission_factor_sources


### 碳排放因子数据来源与说明 (Emission Factor Sources)

**针对审稿人关于表5排放因子无引用来源的回应（P0-3）：**

本研究采用的碳排放因子来源和依据如下：

| 能源载体 | 采用值 (kgCO₂e/kWh) | 数据来源 | 参考年份 |
|----------|---------------------|----------|----------|
| 电力 | 0.55 | 中国生态环境部《企业温室气体排放核算方法与报告指南 发电设施》；华北区域电网平均排放因子 | 2022 |
| 天然气 | 0.202 | GB/T 51366-2019《建筑碳排放计算标准》附录A；天然气低位热值换算 | 2019 |
| 区域供热 | 0.22 | Zheng et al. (2018) Energy and Buildings 179:1-14；北京集中供热系统碳强度 | 2018 |
| 区域供冷 | 0.16 | 基于区域供冷系统典型 COP=4.5 和华北电网电力排放因子反算 (0.55/4.5 + 0.038 输配损耗) | — |

**不确定性说明：**
- 中国电网排放因子持续下降（2015: ~0.60 → 2022: ~0.55 kgCO₂e/kWh），未来随可再生能源占比提升将进一步降低。
- 北京天然气供热占比高于全国平均，天然气排放因子的地区适用性需进一步验证。
- 区域供冷排放因子取决于冷源类型和系统效率，变化范围较大。

**参考文献：**
- 中华人民共和国生态环境部. 企业温室气体排放核算方法与报告指南 发电设施 [S]. 2022.
- GB/T 51366-2019. 建筑碳排放计算标准 [S]. 北京: 中国建筑工业出版社, 2019.
- Zheng, W.; Xu, W.; Wang, D.; et al. A study of city-level building energy efficiency benchmarking system for China. *Energy and Buildings* 2018, 179, 1–14.


<!-- chinese-cell-note -->
#### Cell 04 中文说明

**用途**：读取 Step 1-3 输出，复现共享特征工程，并计算 OCEI、平均碳因子和载体碳排。

**输入与输出**：输入为数据集、SRC 结果和最佳模型清单；输出为带 OCEI 标签的数据表。

**评委回应重点**：它保证碳分析与 EUI 分析使用同一样本和特征定义。


<!-- english-cell-note -->
#### Cell 04 - Data loading and OCEI construction

**Purpose**: This cell loads upstream outputs, recreates shared features, and computes OCEI and carrier-level carbon quantities.

**Inputs and outputs**: Uses the declared emission-factor assumptions to construct OCEI and export coupling evidence.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
# ---------- 2) Load data, build carbon labels, and complete feature engineering ----------
assert DATASET_PATH.exists(), "Please complete Step 1 first"
assert SRC_PATH.exists(), "Please complete Step 2 first"
assert BEST2_PATH.exists() or METRICS_PATH.exists(), "Please complete Step 3 first"

df = pd.read_csv(DATASET_PATH)
src_df = pd.read_csv(SRC_PATH)

if BEST2_PATH.exists():
    best2 = pd.read_csv(BEST2_PATH).iloc[:, 0].dropna().astype(str).tolist()
else:
    metrics_for_best2 = pd.read_csv(METRICS_PATH)
    required_metric_cols = {"model", "test_r2", "test_rmse"}
    missing_metric_cols = required_metric_cols.difference(metrics_for_best2.columns)
    if missing_metric_cols:
        raise ValueError(f"model_metrics.csv is missing required columns: {sorted(missing_metric_cols)}")
    best2 = (
        metrics_for_best2.sort_values(["test_r2", "test_rmse"], ascending=[False, True])
        .head(2)["model"]
        .astype(str)
        .tolist()
    )
    print("best2_models.csv not found; using the top two models from model_metrics.csv:", best2)

# ---------- A) Use feature engineering consistent with Steps 2 and 3 ----------
if "orientation_deg" in df.columns:
    df["orientation_sin"] = np.sin(np.deg2rad(df["orientation_deg"]))
    df["orientation_cos"] = np.cos(np.deg2rad(df["orientation_deg"]))

if "window_type_id" in df.columns:
    df = pd.get_dummies(df, columns=["window_type_id"], prefix="window_type", drop_first=True)

# Ensure that dummy columns exist.
for col in ["window_type_2", "window_type_3"]:
    if col not in df.columns:
        df[col] = 0

# Recalculate footprint_area_m2 and aspect_ratio if they are missing from the raw data.
if "footprint_area_m2" not in df.columns and {"building_length", "building_width"}.issubset(df.columns):
    df["footprint_area_m2"] = df["building_length"] * df["building_width"]

if "aspect_ratio" not in df.columns and {"building_length", "building_width"}.issubset(df.columns):
    df["aspect_ratio"] = df["building_length"] / df["building_width"].replace(0, np.nan)

# ---------- B) Build carbon-emission labels under the fixed accounting boundary ----------
for col in [
    "electricity_kwh", "natural_gas_kwh",
    "district_heating_kwh", "district_cooling_kwh",
    "lighting_electricity_kwh", "equipment_electricity_kwh", "fan_electricity_kwh",
    "ideal_heating_load_kwh", "ideal_cooling_load_kwh", "dhw_thermal_kwh"
]:
    if col not in df.columns:
        df[col] = 0.0

# Fixed boundary:
# Space heating -> district heating.
# Space cooling -> district cooling.
# Domestic hot water -> natural gas.
# Building-side electricity loads -> lighting + equipment + fans.
df["electricity_kwh_for_carbon"] = (
    df["lighting_electricity_kwh"] +
    df["equipment_electricity_kwh"] +
    df["fan_electricity_kwh"]
)

df["natural_gas_kwh_for_carbon"] = df["natural_gas_kwh"]
df["district_heating_kwh_for_carbon"] = df["ideal_heating_load_kwh"]
df["district_cooling_kwh_for_carbon"] = df["ideal_cooling_load_kwh"]

df["carbon_kgco2e"] = (
    df["electricity_kwh_for_carbon"] * EMISSION_FACTORS["electricity"] +
    df["natural_gas_kwh_for_carbon"] * EMISSION_FACTORS["natural_gas"] +
    df["district_heating_kwh_for_carbon"] * EMISSION_FACTORS["district_heating"] +
    df["district_cooling_kwh_for_carbon"] * EMISSION_FACTORS["district_cooling"]
)

df["OCEI_kgco2e_m2"] = df["carbon_kgco2e"] / df["gross_floor_area_m2"]

df["OCEI_electricity"] = (
    df["electricity_kwh_for_carbon"] * EMISSION_FACTORS["electricity"] / df["gross_floor_area_m2"]
)
df["OCEI_natural_gas"] = (
    df["natural_gas_kwh_for_carbon"] * EMISSION_FACTORS["natural_gas"] / df["gross_floor_area_m2"]
)
df["OCEI_district_heating"] = (
    df["district_heating_kwh_for_carbon"] * EMISSION_FACTORS["district_heating"] / df["gross_floor_area_m2"]
)
df["OCEI_district_cooling"] = (
    df["district_cooling_kwh_for_carbon"] * EMISSION_FACTORS["district_cooling"] / df["gross_floor_area_m2"]
)

df["site_energy_kwh_for_carbon"] = (
    df["electricity_kwh_for_carbon"] +
    df["natural_gas_kwh_for_carbon"] +
    df["district_heating_kwh_for_carbon"] +
    df["district_cooling_kwh_for_carbon"]
)

df["carbon_per_kwh"] = df["carbon_kgco2e"] / df["site_energy_kwh_for_carbon"].replace(0, np.nan)

# Guard against invalid denominator or target values before downstream ranking and model validation.
analysis_required_cols = ["eui_kwh_m2", "OCEI_kgco2e_m2", "gross_floor_area_m2"]
missing_analysis_cols = [c for c in analysis_required_cols if c not in df.columns]
if missing_analysis_cols:
    raise KeyError(f"Required analysis columns are missing: {missing_analysis_cols}")

analysis_values = df[analysis_required_cols].replace([np.inf, -np.inf], np.nan)
valid_analysis_mask = analysis_values.notna().all(axis=1) & (df["gross_floor_area_m2"] > 0)
removed_invalid_rows = int((~valid_analysis_mask).sum())
if removed_invalid_rows:
    print(f"Removed {removed_invalid_rows} rows with invalid EUI/OCEI denominators or targets before Step 4 analysis.")

df = df.loc[valid_analysis_mask].reset_index(drop=True)
if len(df) < 10:
    raise ValueError("Fewer than 10 valid cases remain after OCEI quality screening.")

df[[
    "sample_id",
    "eui_kwh_m2",
    "OCEI_kgco2e_m2",
    "carbon_per_kwh",
    "electricity_kwh",
    "natural_gas_kwh"
]].head(99)

<!-- chinese-cell-note -->
#### Cell 03 中文说明

**用途**：构建电力低/高值、电网脱碳和供热变化情景，检验 EUI-OCEI 耦合结论的稳健性。

**输入与输出**：输入为基准排放因子和情景设定；输出为敏感性结果表与图。

**评委回应重点**：它把碳因子不确定性纳入定量分析。


<!-- english-cell-note -->
#### Cell 03 - Emission-factor sensitivity scenarios

**Purpose**: This cell defines emission-factor scenarios to test the robustness of EUI-OCEI coupling under alternative assumptions.

**Inputs and outputs**: Uses the declared emission-factor assumptions to construct OCEI and export coupling evidence.

**Reviewer-facing rationale**: Makes carbon-factor assumptions explicit and tests whether conclusions are robust to plausible factor changes.


In [ ]:
# ============================================================
from scipy.stats import pearsonr
# [IMPROVEMENT P0-3] Carbon Emission Factor Sensitivity Analysis
#
# Test the robustness of EUI-OCEI coupling results under
# alternative emission factor assumptions.
# ============================================================

# Define sensitivity scenarios.
# Grid-decarbonisation values are treated as illustrative stress tests unless the manuscript cites a specific projection source.
scenarios = {
    "Baseline": {"electricity": 0.55, "natural_gas": 0.202, "district_heating": 0.22, "district_cooling": DISTRICT_COOLING_EF_BASELINE, "basis": "Baseline accounting boundary"},
    "Low Electricity (0.40)": {"electricity": 0.40, "natural_gas": 0.202, "district_heating": 0.22, "district_cooling": DISTRICT_COOLING_EF_BASELINE, "basis": "Electricity-factor lower-bound stress test"},
    "High Electricity (0.70)": {"electricity": 0.70, "natural_gas": 0.202, "district_heating": 0.22, "district_cooling": DISTRICT_COOLING_EF_BASELINE, "basis": "Electricity-factor upper-bound stress test"},
    "Grid Decarbonisation 2030 (0.40)": {"electricity": 0.40, "natural_gas": 0.202, "district_heating": 0.22, "district_cooling": DISTRICT_COOLING_EF_BASELINE, "basis": "Illustrative near-term low-carbon grid stress test"},
    "Illustrative Low-Carbon Grid 2050 (0.25)": {"electricity": 0.25, "natural_gas": 0.202, "district_heating": 0.22, "district_cooling": DISTRICT_COOLING_EF_BASELINE, "basis": "Illustrative deep-decarbonisation stress test, not an uncited point prediction"},
    "High COP District Cooling (COP 5.5)": {"electricity": 0.55, "natural_gas": 0.202, "district_heating": 0.22, "district_cooling": 0.55 / 5.5 + DISTRICT_COOLING_AUXILIARY_EF, "basis": "Higher district-cooling system COP sensitivity"},
    "Low COP District Cooling (COP 3.5)": {"electricity": 0.55, "natural_gas": 0.202, "district_heating": 0.22, "district_cooling": 0.55 / 3.5 + DISTRICT_COOLING_AUXILIARY_EF, "basis": "Lower district-cooling system COP sensitivity"},
}

# Compute OCEI under each scenario
scenario_results = []
for sc_name, ef in scenarios.items():
    ocei_sc = (
        df['electricity_kwh_for_carbon'] * ef["electricity"] +
        df['natural_gas_kwh_for_carbon'] * ef["natural_gas"] +
        df['district_heating_kwh_for_carbon'] * ef["district_heating"] +
        df['district_cooling_kwh_for_carbon'] * ef["district_cooling"]
    ) / df['gross_floor_area_m2']

    # Pearson correlation with baseline OCEI
    r_eui, p_eui = pearsonr(df['eui_kwh_m2'], ocei_sc)

    # Top-10% overlap with baseline ranking
    rank_base = df['OCEI_kgco2e_m2'].rank(method='first', ascending=True)
    rank_sc = ocei_sc.rank(method='first', ascending=True)
    top_n = max(1, int(len(df) * 0.10))
    overlap = len(
        set(rank_base.nsmallest(top_n).index) &
        set(rank_sc.nsmallest(top_n).index)
    ) / top_n

    scenario_results.append({
        'scenario': sc_name,
        'scenario_basis': ef.get('basis', ''),
        'electricity_factor': ef["electricity"],
        'district_cooling_factor': ef["district_cooling"],
        'mean_ocei': ocei_sc.mean(),
        'std_ocei': ocei_sc.std(),
        'corr_with_baseline_ocei': ocei_sc.corr(df['OCEI_kgco2e_m2']),
        'eui_ocei_pearson_r': r_eui,
        'eui_ocei_pearson_p': p_eui,
        'top10_overlap_with_baseline': overlap,
    })

scenario_df = pd.DataFrame(scenario_results)

# ============================================================
# Tornado Plot: Sensitivity of mean OCEI to emission factors
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6), dpi=150)

# Panel 1: Mean OCEI across scenarios
ax = axes[0]
baseline_mean = scenario_df.loc[scenario_df['scenario']=='Baseline', 'mean_ocei'].values[0]
colors = ['steelblue' if 'Baseline' in s else 'darkorange' if 'Low' in s or '2030' in s or '2050' in s
          else 'darkred' if 'High' in s else 'grey' for s in scenario_df['scenario']]
bars = ax.barh(scenario_df['scenario'], scenario_df['mean_ocei'], color=colors)
ax.axvline(baseline_mean, color='grey', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xlabel('Mean OCEI [kgCO2e/(m2.a)]', fontsize=12)
ax.set_title('Mean OCEI Under Different Emission Factor Scenarios', fontsize=13)
for bar, val in zip(bars, scenario_df['mean_ocei']):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)
ax.grid(axis='x', alpha=0.3)

# Panel 2: Key metrics stability
ax = axes[1]
metrics_to_plot = ['eui_ocei_pearson_r', 'top10_overlap_with_baseline']
x = np.arange(len(scenario_df))
width = 0.35
bars1 = ax.bar(x - width/2, scenario_df['eui_ocei_pearson_r'], width,
               label='EUI-OCEI Pearson r', color='steelblue')
ax.set_ylabel('Pearson r', fontsize=12, color='steelblue')
ax.tick_params(axis='y', labelcolor='steelblue')

ax2 = ax.twinx()
bars2 = ax2.bar(x + width/2, scenario_df['top10_overlap_with_baseline'] * 100, width,
                label='Top-10% Overlap (%)', color='darkorange')
ax2.set_ylabel('Top-10% Overlap (%)', fontsize=12, color='darkorange')
ax2.tick_params(axis='y', labelcolor='darkorange')

ax.set_xticks(x)
ax.set_xticklabels([s.replace(' (', '\\n(') for s in scenario_df['scenario']],
                    rotation=30, ha='right', fontsize=8)
ax.set_title('Stability of EUI-OCEI Coupling Metrics', fontsize=13)

# Combined legend
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='lower left')

plt.tight_layout()
out_dir = PROJECT_ROOT / 'outputs_step4' / 'figures'
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / 'emission_factor_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()

# Print sensitivity summary
print("=" * 60)
print("EMISSION FACTOR SENSITIVITY ANALYSIS")
print("=" * 60)
print(scenario_df.to_string(index=False))
print(f"\\nBaseline OCEI: {baseline_mean:.2f} ± {scenario_df.loc[scenario_df['scenario']=='Baseline','std_ocei'].values[0]:.2f}")
print(f"Max deviation from baseline: "
      f"{scenario_df['mean_ocei'].max() - baseline_mean:.2f} to "
      f"{scenario_df['mean_ocei'].min() - baseline_mean:.2f} kgCO2e/m2.a")
print(f"EUI-OCEI correlation stability: r range "
      f"[{scenario_df['eui_ocei_pearson_r'].min():.4f}, {scenario_df['eui_ocei_pearson_r'].max():.4f}]")
print(f"Top-10% ranking overlap range: "
      f"[{scenario_df['top10_overlap_with_baseline'].min()*100:.0f}%, "
      f"{scenario_df['top10_overlap_with_baseline'].max()*100:.0f}%]")

# Save
scenario_df.to_csv(PROJECT_ROOT / 'outputs_step4' / 'emission_factor_sensitivity.csv', index=False)


<!-- chinese-cell-note -->
#### Cell 05 中文说明

**用途**：统计并可视化 OCEI 分布，检查碳强度结果的集中趋势、离散程度和范围。

**输入与输出**：输入为 OCEI_kgco2e_m2；输出为 ocei_summary_statistics.csv 和分布图。

**评委回应重点**：它为后续排序、耦合和模型分析建立基准。


<!-- english-cell-note -->
#### Cell 05 - OCEI distribution summary

**Purpose**: This cell summarizes and plots the OCEI distribution for simulated hotel cases.

**Inputs and outputs**: Uses the declared emission-factor assumptions to construct OCEI and export coupling evidence.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
# ---------- 2A) OCEI distribution and statistics ----------
ocei_stats = pd.DataFrame([{
    "mean_ocei": df["OCEI_kgco2e_m2"].mean(),
    "median_ocei": df["OCEI_kgco2e_m2"].median(),
    "std_ocei": df["OCEI_kgco2e_m2"].std(),
    "min_ocei": df["OCEI_kgco2e_m2"].min(),
    "max_ocei": df["OCEI_kgco2e_m2"].max(),
}])

ocei_stats.to_csv(OUT_DIR / "ocei_summary_statistics.csv", index=False, encoding="utf-8-sig")
display(ocei_stats.round(4))

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.hist(df["OCEI_kgco2e_m2"].dropna(), bins=30, edgecolor="black", alpha=0.75)
ax.axvline(df["OCEI_kgco2e_m2"].mean(), linestyle="--", linewidth=1.5, label=f"Mean = {df['OCEI_kgco2e_m2'].mean():.2f}",color = 'orange')
ax.axvline(df["OCEI_kgco2e_m2"].median(), linestyle="-.", linewidth=1.5, label=f"Median = {df['OCEI_kgco2e_m2'].median():.2f}", color ='red')
ax.set_title("Distribution of OCEI")
ax.set_xlabel("OCEI (kgCO2e/m²·year)")
ax.set_ylabel("Frequency")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(FIG_DIR / "ocei_distribution.png", bbox_inches="tight")
plt.show()

<!-- chinese-cell-note -->
#### Cell 06 中文说明

**用途**：封装不同情景下 OCEI 数据构建流程，保证敏感性分析使用一致的数据处理口径。

**输入与输出**：输入为数据路径、情景名称、供热口径和排放因子；输出为情景化碳数据表。

**评委回应重点**：它降低重复代码导致的碳核算口径漂移。


<!-- english-cell-note -->
#### Cell 06 - Reusable carbon-dataset builder

**Purpose**: This cell defines a reusable function for constructing carbon datasets under alternative scenarios.

**Inputs and outputs**: Uses the declared emission-factor assumptions to construct OCEI and export coupling evidence.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
# ---------- 2B) Multi-scenario OCEI calculation function ----------
def build_carbon_dataset(dataset_path, scenario="baseline", heat_mode="space_only", emission_factors=None):
    if emission_factors is None:
        emission_factors = EMISSION_FACTORS

    d = pd.read_csv(dataset_path)

    # Use feature engineering consistent with the earlier steps.
    if "orientation_deg" in d.columns:
        d["orientation_sin"] = np.sin(np.deg2rad(d["orientation_deg"]))
        d["orientation_cos"] = np.cos(np.deg2rad(d["orientation_deg"]))

    if "window_type_id" in d.columns:
        d = pd.get_dummies(d, columns=["window_type_id"], prefix="window_type", drop_first=True)

    for col in ["window_type_2", "window_type_3"]:
        if col not in d.columns:
            d[col] = 0

    if "footprint_area_m2" not in d.columns and {"building_length", "building_width"}.issubset(d.columns):
        d["footprint_area_m2"] = d["building_length"] * d["building_width"]

    if "aspect_ratio" not in d.columns and {"building_length", "building_width"}.issubset(d.columns):
        d["aspect_ratio"] = d["building_length"] / d["building_width"].replace(0, np.nan)

    for col in [
        "electricity_kwh", "natural_gas_kwh",
        "district_heating_kwh", "district_cooling_kwh",
        "lighting_electricity_kwh", "equipment_electricity_kwh", "fan_electricity_kwh",
        "ideal_heating_load_kwh", "ideal_cooling_load_kwh", "dhw_thermal_kwh"
    ]:
        if col not in d.columns:
            d[col] = 0.0

    if scenario == "baseline":
        d["electricity_kwh_for_carbon"] = d["electricity_kwh"]
        d["natural_gas_kwh_for_carbon"] = d["natural_gas_kwh"]
        d["district_heating_kwh_for_carbon"] = d["district_heating_kwh"]
        d["district_cooling_kwh_for_carbon"] = d["district_cooling_kwh"]

    elif scenario == "district":
        d["electricity_kwh_for_carbon"] = (
            d["lighting_electricity_kwh"] +
            d["equipment_electricity_kwh"] +
            d["fan_electricity_kwh"]
        )
        d["district_cooling_kwh_for_carbon"] = d["ideal_cooling_load_kwh"]

        if heat_mode == "space_plus_dhw":
            d["natural_gas_kwh_for_carbon"] = 0.0
            d["district_heating_kwh_for_carbon"] = d["ideal_heating_load_kwh"] + d["dhw_thermal_kwh"]
        elif heat_mode == "space_only":
            d["natural_gas_kwh_for_carbon"] = d["natural_gas_kwh"]
            d["district_heating_kwh_for_carbon"] = d["ideal_heating_load_kwh"]
        else:
            raise ValueError(f"Unknown heat_mode: {heat_mode}")
    else:
        raise ValueError(f"Unknown scenario: {scenario}")

    d["carbon_kgco2e"] = (
        d["electricity_kwh_for_carbon"] * emission_factors["electricity"] +
        d["natural_gas_kwh_for_carbon"] * emission_factors["natural_gas"] +
        d["district_heating_kwh_for_carbon"] * emission_factors["district_heating"] +
        d["district_cooling_kwh_for_carbon"] * emission_factors["district_cooling"]
    )

    d["OCEI_kgco2e_m2"] = d["carbon_kgco2e"] / d["gross_floor_area_m2"]

    d["site_energy_kwh_for_carbon"] = (
        d["electricity_kwh_for_carbon"] +
        d["natural_gas_kwh_for_carbon"] +
        d["district_heating_kwh_for_carbon"] +
        d["district_cooling_kwh_for_carbon"]
    )

    d["carbon_per_kwh"] = d["carbon_kgco2e"] / d["site_energy_kwh_for_carbon"].replace(0, np.nan)

    d["scenario_name"] = scenario if scenario == "baseline" else f"{scenario}_{heat_mode}"
    return d

<!-- chinese-cell-note -->
#### Cell 07 中文说明

**用途**：重建第三部分前两名模型结构，并应用于 OCEI 预测任务。

**输入与输出**：输入为前 18 个变量、模型参数和 OCEI 标签；输出为 models_for_carbon。

**评委回应重点**：它检验 EUI 代理建模框架能否迁移到碳强度指标。


<!-- english-cell-note -->
#### Cell 07 - Carbon-model reconstruction

**Purpose**: This cell rebuilds the leading Step 3 model structures for OCEI prediction.

**Inputs and outputs**: Fits the declared model family, stores metrics, and exports reusable model artefacts.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
# ---------- 3) Load the top 18 variables and rebuild the top two models ----------
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor

try:
    from xgboost import XGBRegressor
    HAS_XGB = True
except Exception:
    HAS_XGB = False

try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False

top18 = src_df.sort_values("abs_SRC", ascending=False).head(18)["feature"].tolist()

missing = [c for c in top18 if c not in df.columns]
if missing:
    raise KeyError(f"Top18 features missing after feature engineering: {missing}")

X = df[top18].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).copy()
y = pd.to_numeric(df["OCEI_kgco2e_m2"], errors="coerce").replace([np.inf, -np.inf], np.nan)
valid_target_mask = y.notna()
if not valid_target_mask.all():
    removed_targets = int((~valid_target_mask).sum())
    print(f"Removed {removed_targets} rows with non-finite OCEI targets before carbon-model validation.")
    X = X.loc[valid_target_mask].reset_index(drop=True)
    y = y.loc[valid_target_mask].reset_index(drop=True)
else:
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

prep_linear = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

prep_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

alphas = np.logspace(-4, 4, 40)

models = {
    "Linear": Pipeline([
        ("prep", prep_linear),
        ("model", LinearRegression())
    ]),

    "RidgeCV": Pipeline([
        ("prep", prep_linear),
        ("model", RidgeCV(alphas=alphas, cv=10))
    ]),

    "LassoCV": Pipeline([
        ("prep", prep_linear),
        ("model", LassoCV(alphas=alphas, cv=10, max_iter=50000, random_state=42))
    ]),

    "ElasticNetCV": Pipeline([
        ("prep", prep_linear),
        ("model", ElasticNetCV(
            l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
            alphas=alphas,
            cv=10,
            max_iter=50000,
            random_state=42
        ))
    ]),

    "Poly2-RidgeCV": Pipeline([
        ("prep", prep_linear),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("poly_scaler", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-2, 4, 30), cv=10))
    ]),

    "Poly2-ElasticNetCV": Pipeline([
        ("prep", prep_linear),
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("poly_scaler", StandardScaler()),
        ("model", ElasticNetCV(
            l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9],
            alphas=np.logspace(-3, 2, 20),
            cv=10,
            max_iter=50000,
            random_state=42
        ))
    ]),

    "Poly2-Interaction-RidgeCV": Pipeline([
        ("prep", prep_linear),
        ("poly", PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)),
        ("poly_scaler", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-2, 4, 30), cv=10))
    ]),

    "Poly3-RidgeCV": Pipeline([
        ("prep", prep_linear),
        ("poly", PolynomialFeatures(degree=3, include_bias=False)),
        ("poly_scaler", StandardScaler()),
        ("model", RidgeCV(alphas=np.logspace(-1, 5, 30), cv=10))
    ]),

    "KNN": Pipeline([
        ("prep", prep_linear),
        ("model", KNeighborsRegressor(n_neighbors=8, weights="distance"))
    ]),

    "SVR-RBF": Pipeline([
        ("prep", prep_linear),
        ("model", SVR(C=20.0, epsilon=0.02, kernel="rbf", gamma="scale"))
    ]),

    "DecisionTree": Pipeline([
        ("prep", prep_tree),
        ("model", DecisionTreeRegressor(max_depth=8, min_samples_leaf=4, random_state=42))
    ]),

    "RandomForest": Pipeline([
        ("prep", prep_tree),
        ("model", RandomForestRegressor(
            n_estimators=800,
            max_depth=8,
            min_samples_leaf=2,
            min_samples_split=4,
            n_jobs=1,
            random_state=42
        ))
    ]),

    "ExtraTrees": Pipeline([
        ("prep", prep_tree),
        ("model", ExtraTreesRegressor(
            n_estimators=800,
            max_depth=8,
            min_samples_leaf=2,
            min_samples_split=4,
            n_jobs=1,
            random_state=42
        ))
    ]),

    "GB": Pipeline([
        ("prep", prep_tree),
        ("model", GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=3,
            random_state=42
        ))
    ]),

    "MLP": Pipeline([
        ("prep", prep_linear),
        ("model", MLPRegressor(
            hidden_layer_sizes=(64, 32),
            alpha=1e-3,
            random_state=42,
            max_iter=5000
        ))
    ]),
}

if HAS_XGB:
    models["XGBoost"] = Pipeline([
        ("prep", prep_tree),
        ("model", XGBRegressor(
            n_estimators=400,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=1
        ))
    ])

if HAS_LGBM:
    models["LightGBM"] = Pipeline([
        ("prep", prep_tree),
        ("model", LGBMRegressor(
            n_estimators=400,
            learning_rate=0.05,
            num_leaves=15,
            min_child_samples=10,
            subsample=0.9,
            colsample_bytree=0.9,
            random_state=42,
            verbose=-1
        ))
    ])

models_for_carbon = {name: models[name] for name in best2 if name in models}

print("Best2 from step3:", best2)
print("Available models:", list(models.keys()))
print("Carbon models selected:", list(models_for_carbon.keys()))

<!-- chinese-cell-note -->
#### Cell 08 中文说明

**用途**：用 10-fold 交叉验证评估前两名模型预测 OCEI 的能力，并记录误差与稳定性。

**输入与输出**：输入为 OCEI 特征矩阵和模型字典；输出为 carbon_model_metrics.csv 与折外预测。

**评委回应重点**：它说明碳强度是否也能由关键变量可靠解释。


<!-- english-cell-note -->
#### Cell 08 - OCEI surrogate-model evaluation

**Purpose**: This cell evaluates the leading models for OCEI prediction using 10-fold cross-validation.

**Inputs and outputs**: Fits the declared model family, stores metrics, and exports reusable model artefacts.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
# ---------- 4) Evaluate carbon-intensity prediction by the top two models ----------
cv = KFold(n_splits=10, shuffle=True, random_state=42)
carbon_metrics = []
carbon_oof = {}

for name, estimator in models_for_carbon.items():
    oof = np.full(len(X), np.nan)
    fold_r2_scores = []

    for tr, te in cv.split(X):
        est = clone(estimator)
        est.fit(X.iloc[tr], y.iloc[tr])
        pred_fold = est.predict(X.iloc[te])

        oof[te] = pred_fold
        fold_r2_scores.append(r2_score(y.iloc[te], pred_fold))

    carbon_oof[name] = oof
    carbon_metrics.append({
        "model": name,
        "R2": r2_score(y, oof),
        "cv_r2_mean": np.mean(fold_r2_scores),
        "cv_r2_std": np.std(fold_r2_scores, ddof=1),
        "cv_r2_variance": np.var(fold_r2_scores, ddof=1),
        "RMSE": np.sqrt(mean_squared_error(y, oof)),
        "MAE": mean_absolute_error(y, oof),
        "MAPE": mean_absolute_percentage_error(y, oof),
    })

carbon_metrics_df = pd.DataFrame(carbon_metrics)
carbon_metrics_df = carbon_metrics_df.sort_values(["R2", "cv_r2_variance"], ascending=[False, True]).reset_index(drop=True)
carbon_metrics_df.to_csv(OUT_DIR / "carbon_model_metrics.csv", index=False, encoding="utf-8-sig")
carbon_metrics_df

<!-- chinese-cell-note -->
#### Cell 09 中文说明

**用途**：按能源载体汇总年度碳排放贡献，识别电力、天然气、供热和供冷的相对作用。

**输入与输出**：输入为载体能耗和排放因子；输出为载体贡献图。

**评委回应重点**：它支撑工程意义和减排优先级讨论。


<!-- english-cell-note -->
#### Cell 09 - Carrier contribution overview

**Purpose**: This cell aggregates annual carbon emissions by carrier to identify dominant contributors.

**Inputs and outputs**: Uses documented upstream objects and writes the files or in-memory tables needed by the next notebook cell.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:

# ---------- 5) Carbon-emission contribution breakdown figure ----------
contrib = pd.Series({
    "electricity": (df["electricity_kwh_for_carbon"] * EMISSION_FACTORS["electricity"]).sum(),
    "natural_gas": (df["natural_gas_kwh_for_carbon"] * EMISSION_FACTORS["natural_gas"]).sum(),
    "district_heating": (df["district_heating_kwh_for_carbon"] * EMISSION_FACTORS["district_heating"]).sum(),
    "district_cooling": (df["district_cooling_kwh_for_carbon"] * EMISSION_FACTORS["district_cooling"]).sum(),
}).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(7.4, 4.8))
ax.bar(contrib.index, contrib.values)
ax.set_title("Contribution of carbon emissions by Energy Carrier")
ax.set_ylabel("Annual carbon contribution (kgCO2e.year)")
fig.tight_layout()
fig.savefig(FIG_DIR / "carbon_contribution_by_carrier.png", bbox_inches="tight")
plt.show()

<!-- chinese-cell-note -->
#### 合并碳贡献图说明

**用途**：把年度总碳排放贡献和单位面积 OCEI 贡献放在同一张双面板图中，回应审稿人关于合并图和图形可读性的意见。

**输入与输出**：输入为 Step 4 已构建的四类碳载体能耗；输出为 `carbon_contribution_consolidated.png`。

**评委回应重点**：它使碳贡献结构可以直接比较，同时保持排放载体边界清晰。


<!-- english-cell-note -->
#### Consolidated Carbon-Contribution Figure

**Purpose**: This cell combines total annual carbon contribution and area-normalised OCEI contribution in one dual-panel figure.

**Inputs and outputs**: It uses the four carbon-accounting carrier columns generated in Step 4 and exports `carbon_contribution_consolidated.png`.

**Reviewer-facing rationale**: The figure makes the carrier-level carbon structure easier to audit while preserving the accounting boundary.


In [ ]:
# [R1-MIN-5] Consolidated dual-panel carbon contribution figure
carrier_order = ["electricity", "natural_gas", "district_heating", "district_cooling"]
carrier_colors = {
    "electricity": "#4C72B0",
    "natural_gas": "#DD8452",
    "district_heating": "#55A868",
    "district_cooling": "#C44E52",
}
carrier_labels = {
    "electricity": "Electricity",
    "natural_gas": "Natural Gas",
    "district_heating": "District Heating",
    "district_cooling": "District Cooling",
}
carrier_energy_cols = {
    "electricity": "electricity_kwh_for_carbon",
    "natural_gas": "natural_gas_kwh_for_carbon",
    "district_heating": "district_heating_kwh_for_carbon",
    "district_cooling": "district_cooling_kwh_for_carbon",
}

carrier_carbon_total = {
    carrier: float((df[carrier_energy_cols[carrier]] * EMISSION_FACTORS[carrier]).sum())
    for carrier in carrier_order
}

if "gross_floor_area_m2" in df.columns:
    area_denominator = df["gross_floor_area_m2"]
elif "total_floor_area_m2" in df.columns:
    area_denominator = df["total_floor_area_m2"]
elif {"footprint_area_m2", "floor_num"}.issubset(df.columns):
    area_denominator = df["footprint_area_m2"] * df["floor_num"]
else:
    raise KeyError("No floor-area denominator found for OCEI contribution calculation.")

area_denominator = pd.to_numeric(area_denominator, errors="coerce").replace(0, np.nan)
carrier_ocei_avg = {
    carrier: float((df[carrier_energy_cols[carrier]] * EMISSION_FACTORS[carrier] / area_denominator).mean())
    for carrier in carrier_order
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), dpi=150)

ax = axes[0]
totals = [carrier_carbon_total[c] for c in carrier_order]
bars = ax.bar([carrier_labels[c] for c in carrier_order], totals, color=[carrier_colors[c] for c in carrier_order])
ax.set_title("(a) Total Annual Carbon by Carrier")
ax.set_ylabel("kgCO2e / year")
ax.tick_params(axis="x", rotation=25)
for bar, val in zip(bars, totals):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(totals) * 0.01, f"{val / 1e6:.1f}M", ha="center", fontsize=9)

ax = axes[1]
avg_ocei = [carrier_ocei_avg[c] for c in carrier_order]
bars = ax.bar([carrier_labels[c] for c in carrier_order], avg_ocei, color=[carrier_colors[c] for c in carrier_order])
ax.set_title("(b) Average OCEI Contribution by Carrier")
ax.set_ylabel("kgCO2e / (m2.year)")
ax.tick_params(axis="x", rotation=25)
for bar, val in zip(bars, avg_ocei):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(avg_ocei) * 0.01, f"{val:.2f}", ha="center", fontsize=9)

fig.tight_layout()
fig.savefig(FIG_DIR / "carbon_contribution_consolidated.png", dpi=300, bbox_inches="tight")
plt.show()

<!-- chinese-cell-note -->
#### Cell 10 中文说明

**用途**：同时展示总量贡献和单位面积贡献，合并原本分散的碳贡献表达。

**输入与输出**：输入为各载体年度碳排总量；输出为合并贡献图和汇总表。

**评委回应重点**：它回应图表冗余和可读性问题。


<!-- english-cell-note -->
#### Cell 10 - Dual-view carbon contribution

**Purpose**: This cell combines absolute and floor-area-normalized carbon contributions into one dual-view figure.

**Inputs and outputs**: Uses the declared emission-factor assumptions to construct OCEI and export coupling evidence.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
carrier_carbon_total = pd.Series({
    "electricity": (df["electricity_kwh_for_carbon"] * EMISSION_FACTORS["electricity"]).sum(),
    "natural_gas": (df["natural_gas_kwh_for_carbon"] * EMISSION_FACTORS["natural_gas"]).sum(),
}, dtype=float)

# Include a column only when it exists.
if "district_heating_kwh_for_carbon" in df.columns:
    carrier_carbon_total["district_heating"] = (df["district_heating_kwh_for_carbon"] * EMISSION_FACTORS["district_heating"]).sum()
if "district_cooling_kwh_for_carbon" in df.columns:
    carrier_carbon_total["district_cooling"] = (df["district_cooling_kwh_for_carbon"] * EMISSION_FACTORS["district_cooling"]).sum()

carrier_order = ["electricity", "natural_gas", "district_heating", "district_cooling"]
carrier_carbon_total = carrier_carbon_total.reindex(carrier_order).fillna(0.0)
carrier_carbon_avg = carrier_carbon_total / len(df)
carrier_share = carrier_carbon_total / carrier_carbon_total.sum()

carbon_breakdown_df = pd.DataFrame({
    "carrier": carrier_carbon_total.index,
    "total_carbon_kgco2e": carrier_carbon_total.values,
    "avg_carbon_kgco2e_per_sample": carrier_carbon_avg.values,
    "share": carrier_share.values,
})
carbon_breakdown_df.to_csv(OUT_DIR / "carbon_breakdown_by_carrier.csv", index=False, encoding="utf-8-sig")
display(carbon_breakdown_df)

<!-- chinese-cell-note -->
#### Cell 11 中文说明

**用途**：计算各载体对平均 OCEI 的贡献，解释同一 EUI 下碳强度可能不同的原因。

**输入与输出**：输入为载体能耗、排放因子和建筑面积；输出为载体 OCEI 分解图。

**评委回应重点**：它揭示能源结构如何调节 EUI 与 OCEI 的关系。


<!-- english-cell-note -->
#### Cell 11 - Average carrier-specific OCEI

**Purpose**: This cell computes average carrier-specific OCEI contributions and plots the normalized breakdown.

**Inputs and outputs**: Uses the declared emission-factor assumptions to construct OCEI and export coupling evidence.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
carrier_ocei_avg = pd.Series({
    "electricity": (
        df["electricity_kwh_for_carbon"] * EMISSION_FACTORS["electricity"] / df["gross_floor_area_m2"]
    ).mean(),
    "natural_gas": (
        df["natural_gas_kwh_for_carbon"] * EMISSION_FACTORS["natural_gas"] / df["gross_floor_area_m2"]
    ).mean(),
    "district_heating": (
        df["district_heating_kwh_for_carbon"] * EMISSION_FACTORS["district_heating"] / df["gross_floor_area_m2"]
    ).mean(),
    "district_cooling": (
        df["district_cooling_kwh_for_carbon"] * EMISSION_FACTORS["district_cooling"] / df["gross_floor_area_m2"]
    ).mean(),
}, dtype=float)

carrier_order = ["electricity", "natural_gas", "district_heating", "district_cooling"]
carrier_ocei_avg = carrier_ocei_avg.reindex(carrier_order).fillna(0.0)

fig, ax = plt.subplots(figsize=(7.2, 4.8))
ax.bar(carrier_ocei_avg.index, carrier_ocei_avg.values)
ax.set_title("Contribution of average OCEI by Energy Carrier")
ax.set_ylabel("kgCO2e/m²·year")
fig.tight_layout()
fig.savefig(FIG_DIR / "ocei_breakdown_by_carrier.png", bbox_inches="tight")
plt.show()

<!-- chinese-cell-note -->
#### Cell 12 中文说明

**用途**：计算 EUI 与 OCEI 的相关性，并绘制耦合散点图。

**输入与输出**：输入为 eui_kwh_m2、OCEI_kgco2e_m2 和 carbon_per_kwh；输出为相关系数和散点图。

**评委回应重点**：它提供能耗指标能否代表碳强度的核心统计证据。


<!-- english-cell-note -->
#### Cell 12 - EUI-OCEI correlation

**Purpose**: This cell quantifies and plots the EUI-OCEI relationship across simulated cases.

**Inputs and outputs**: Uses the declared emission-factor assumptions to construct OCEI and export coupling evidence.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
# ---------- EUI-OCEI relationship ----------
from scipy.stats import pearsonr

r_eui_ocei, p_eui_ocei = pearsonr(df["eui_kwh_m2"], df["OCEI_kgco2e_m2"])
r_carbon_factor, p_carbon_factor = pearsonr(df["carbon_per_kwh"], df["OCEI_kgco2e_m2"])

fig, ax = plt.subplots(figsize=(6.8, 6.0))
ax.scatter(df["eui_kwh_m2"], df["OCEI_kgco2e_m2"], alpha=0.25)
ax.set_xlabel("EUI (kWh/m²·year)")
ax.set_ylabel("OCEI (kgCO2e/m²·year)")
ax.set_title("Cross-analysis of EUI and OCEI")
fig.tight_layout()
fig.savefig(FIG_DIR / "eui_ocei_scatter.png", bbox_inches="tight")
plt.show()

print(f"Pearson correlation between EUI and OCEI: r = {r_eui_ocei:.4f}, p = {p_eui_ocei:.4e}")
print(f"Pearson correlation between carbon_per_kwh and OCEI: r = {r_carbon_factor:.4f}, p = {p_carbon_factor:.4e}")

<!-- chinese-cell-note -->
#### Cell 13 中文说明

**用途**：比较 EUI 排序和 OCEI 排序，识别低能耗但不一定低碳的样本。

**输入与输出**：输入为 EUI 与 OCEI；输出为排序偏移表、Top-10% 重叠率和偏移图。

**评委回应重点**：它证明单一 EUI 指标不足以支撑碳优化决策。


<!-- english-cell-note -->
#### Cell 13 - Ranking-shift analysis

**Purpose**: This cell compares EUI-based and OCEI-based rankings to reveal ranking shifts.

**Inputs and outputs**: Uses documented upstream objects and writes the files or in-memory tables needed by the next notebook cell.

**Reviewer-facing rationale**: Quantifies possible feasibility-screening bias and makes accepted-case filtering auditable.


In [ ]:
# ---------- 5A) Ranking shift analysis between EUI and OCEI ----------
rank_df = df[["eui_kwh_m2", "OCEI_kgco2e_m2"]].copy()

rank_df["rank_eui"] = rank_df["eui_kwh_m2"].rank(method="first", ascending=True)
rank_df["rank_ocei"] = rank_df["OCEI_kgco2e_m2"].rank(method="first", ascending=True)
rank_df["rank_shift"] = rank_df["rank_ocei"] - rank_df["rank_eui"]
rank_df["abs_rank_shift"] = rank_df["rank_shift"].abs()

top_n = max(1, int(len(rank_df) * 0.10))
top_eui_idx = set(rank_df.nsmallest(top_n, "eui_kwh_m2").index)
top_ocei_idx = set(rank_df.nsmallest(top_n, "OCEI_kgco2e_m2").index)
overlap_ratio = len(top_eui_idx & top_ocei_idx) / top_n

rank_summary = pd.DataFrame([{
    "sample_count": len(rank_df),
    "top_10pct_n": top_n,
    "top10_overlap_ratio": overlap_ratio,
    "mean_abs_rank_shift": rank_df["abs_rank_shift"].mean(),
    "median_abs_rank_shift": rank_df["abs_rank_shift"].median(),
    "max_abs_rank_shift": rank_df["abs_rank_shift"].max(),
}])

rank_summary.to_csv(OUT_DIR / "eui_ocei_rank_shift_summary.csv", index=False, encoding="utf-8-sig")
rank_df.to_csv(OUT_DIR / "eui_ocei_rank_shift_detail.csv", index=False, encoding="utf-8-sig")

display(rank_summary.round(4))

fig, ax = plt.subplots(figsize=(6.6, 6.0))
ax.scatter(rank_df["rank_eui"], rank_df["rank_ocei"], alpha=0.35)
ax.plot([1, len(rank_df)], [1, len(rank_df)], color="red", linewidth=1.2)
ax.set_xlabel("Rank by EUI")
ax.set_ylabel("Rank by OCEI")
ax.set_title("Ranking Shift between EUI and OCEI")
fig.tight_layout()
fig.savefig(FIG_DIR / "eui_ocei_rank_shift.png", bbox_inches="tight")
plt.show()

<!-- chinese-cell-note -->
#### Cell 14 中文说明

**用途**：构建 EUI 与 OCEI 敏感性比较的扩展变量列表，纳入热水、效率和运行因素。

**输入与输出**：输入为前序特征工程结果；输出为 top20 特征集合。

**评委回应重点**：它避免碳分析被过窄的 EUI 特征集合限制。


<!-- english-cell-note -->
#### Cell 14 - Extended factor list for comparison

**Purpose**: This cell defines an extended factor list for EUI-OCEI sensitivity comparison.

**Inputs and outputs**: Uses documented upstream objects and writes the files or in-memory tables needed by the next notebook cell.

**Reviewer-facing rationale**: Makes carbon-factor assumptions explicit and tests whether conclusions are robust to plausible factor changes.


In [ ]:
# ---------- 5B-0) Extend to 20 input variables for EUI-OCEI factor comparison ----------
top20 = [
    "dhw_per_person",
    "floor_num",
    "room_count",
    "footprint_area_m2",
    "dhw_temp",
    "boiler_eff",
    "cop_heating",
    "operation_hours",
    "fresh_air_ach",
    "light_power",
    "wwr",
    "cop_cooling",
    "floor_height",
    "aspect_ratio",
    "equip_power",
    "heat_set",
    "room_area",
    "u_wall",
    "shgc_s",
    "orientation_sin",
]

<!-- chinese-cell-note -->
#### Cell 15 中文说明

**用途**：分别对 EUI 与 OCEI 运行 Bootstrap SRC，比较两类指标的驱动因素。

**输入与输出**：输入为扩展变量列表、EUI 和 OCEI；输出为 eui_ocei_factor_compare_bootstrap_src.csv。

**评委回应重点**：它揭示能耗优化与碳优化优先级是否一致。


<!-- english-cell-note -->
#### Cell 15 - Bootstrap SRC for EUI and OCEI

**Purpose**: This cell runs bootstrap SRC for both EUI and OCEI and prepares a comparative sensitivity table.

**Inputs and outputs**: Computes SRC estimates, uncertainty summaries, and ranked sensitivity evidence for manuscript reporting.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
from tools.sensitivity_utils import run_src_model

compare_features = [c for c in top20 if c in df.columns]

src_eui_cmp = run_src_model(df, compare_features, "eui_kwh_m2", seed=42, B=1000)
src_ocei_cmp = run_src_model(df, compare_features, "OCEI_kgco2e_m2", seed=42, B=1000)

factor_compare_df = (
    src_eui_cmp.rename(columns={
        "SRC": "EUI_SRC",
        "abs_SRC": "abs_EUI_SRC",
        "CI_low": "EUI_CI_low",
        "CI_high": "EUI_CI_high",
        "sign_stable": "EUI_sign_stable"
    })
    .merge(
        src_ocei_cmp.rename(columns={
            "SRC": "OCEI_SRC",
            "abs_SRC": "abs_OCEI_SRC",
            "CI_low": "OCEI_CI_low",
            "CI_high": "OCEI_CI_high",
            "sign_stable": "OCEI_sign_stable"
        }),
        on="feature",
        how="inner"
    )
)

factor_compare_df["delta_abs"] = factor_compare_df["abs_OCEI_SRC"] - factor_compare_df["abs_EUI_SRC"]
factor_compare_df["max_abs"] = factor_compare_df[["abs_EUI_SRC", "abs_OCEI_SRC"]].max(axis=1)

factor_compare_df = factor_compare_df.sort_values("max_abs", ascending=False)
factor_compare_df.to_csv(OUT_DIR / "eui_ocei_factor_compare_bootstrap_src.csv", index=False, encoding="utf-8-sig")
display(factor_compare_df.head(20).round(4))

<!-- chinese-cell-note -->
#### Cell 16 中文说明

**用途**：绘制 EUI 与 OCEI 关键因素并列图，突出敏感性结构的相同点和差异。

**输入与输出**：输入为 factor_compare_df；输出为关键因素对比图。

**评委回应重点**：它服务于论文双指标框架的结果表达。


<!-- english-cell-note -->
#### Cell 16 - EUI-OCEI factor comparison figure

**Purpose**: This cell visualizes side-by-side absolute SRC values for EUI and OCEI key factors.

**Inputs and outputs**: Reads the validated step outputs and writes English-only manuscript figures to the documented figures folder.

**Reviewer-facing rationale**: Makes carbon-factor assumptions explicit and tests whether conclusions are robust to plausible factor changes.


In [ ]:
# ---------- 5B) Key-factor comparison between EUI and OCEI ----------
plot_df = factor_compare_df.head(20).iloc[::-1]

fig, ax = plt.subplots(figsize=(8.6, 6.8))
ypos = np.arange(len(plot_df))

ax.barh(ypos - 0.18, plot_df["abs_EUI_SRC"], height=0.35, label="EUI")
ax.barh(ypos + 0.18, plot_df["abs_OCEI_SRC"], height=0.35, label="OCEI")

ax.set_yticks(ypos)
ax.set_yticklabels(plot_df["feature"])
ax.set_xlabel("Absolute standardized regression coefficient (|SRC|)")
ax.set_title("Comparison of Key Factors for EUI and OCEI")
ax.legend(frameon=False)

fig.tight_layout()
fig.savefig(FIG_DIR / "eui_ocei_factor_compare.png", bbox_inches="tight")
plt.show()

<!-- chinese-cell-note -->
#### Cell 17 中文说明

**用途**：绘制 OCEI 折外预测值与模拟值对比图，并标注误差指标。

**输入与输出**：输入为 carbon_oof、真实 OCEI 和模型指标；输出为 OCEI 预测散点图。

**评委回应重点**：它完成从 EUI 代理验证到碳强度代理验证的证据链。


<!-- english-cell-note -->
#### Cell 17 - Predicted-versus-simulated OCEI plots

**Purpose**: This cell plots out-of-fold predicted versus simulated OCEI for the leading carbon surrogate models.

**Inputs and outputs**: Reads the validated step outputs and writes English-only manuscript figures to the documented figures folder.

**Reviewer-facing rationale**: Documents the methodological reason for the computation so reviewers can trace each claim to a reproducible output.


In [ ]:
# ---------- 6) Top two models: predicted versus simulated carbon intensity ----------
for name, pred in carbon_oof.items():
    valid = np.isfinite(pred)
    fig, ax = plt.subplots(figsize=(6.8, 6.2))
    ax.scatter(
        y[valid], pred[valid],
        s=12,
        color="#4C72B0",
        alpha=0.25,
        edgecolors="none",
        rasterized=True
    )
    lo = min(y.min(), np.nanmin(pred))
    hi = max(y.max(), np.nanmax(pred))
    ax.plot([lo, hi], [lo, hi], linewidth=1.2, color='red')
    ax.set_title(f"{name}: Predicted vs Simulated Carbon Intensity")
    ax.set_xlabel("Simulated carbon intensity (kgCO2e/m²·year)")
    ax.set_ylabel("Predicted carbon intensity (kgCO2e/m²·year)")
    row = carbon_metrics_df.loc[carbon_metrics_df["model"] == name].iloc[0]

    txt = (
        f"R² = {r2_score(y[valid], pred[valid]):.4f}\n"
        f"CV Var(R²) = {row['cv_r2_variance']:.6f}\n"
        f"RMSE = {np.sqrt(mean_squared_error(y[valid], pred[valid])):.4f}\n"
        f"MAE = {mean_absolute_error(y[valid], pred[valid]):.4f}\n"
        f"MAPE = {mean_absolute_percentage_error(y[valid], pred[valid]):.4f}"
    )
    ax.text(0.03, 0.97, txt, transform=ax.transAxes, va="top", ha="left",
            bbox=dict(boxstyle="round,pad=0.35", facecolor="white", alpha=0.85))
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}_carbon_pred_vs_sim.png", bbox_inches="tight")
    plt.show()


<!-- chinese-cell-note -->
#### 单设计定量案例说明

**用途**：从全真模拟样本中选择接近中位 EUI 的酒店方案，演示“18 个设计参数 → 代理模型 → EUI/OCEI 解释”的完整应用链路。

**输入与输出**：输入为 Step 3 最佳 EUI 模型、top-18 变量和 Step 4 碳核算结果；输出为 `quantitative_use_case.csv`。

**评委回应重点**：它把方法从统计模型转化为可复现的单案例设计流程，说明模型如何服务早期设计决策。


<!-- english-cell-note -->
#### Quantitative Single-Design Walkthrough

**Purpose**: This cell selects a median-EUI hotel case and demonstrates the complete chain from 18 design parameters to surrogate EUI prediction and OCEI interpretation.

**Inputs and outputs**: It uses the Step 3 best EUI model, the selected top-18 variables, and the Step 4 carbon-accounting table, then exports `quantitative_use_case.csv`.

**Reviewer-facing rationale**: The case translates the workflow into an auditable design-use example rather than leaving the contribution at aggregate model metrics only.


In [ ]:
# [R3-9] Quantitative use-case: median-EUI hotel design walkthrough
import joblib

model_path = PROJECT_ROOT / "outputs_step3" / "models" / "Poly3-RidgeCV_eui_model.joblib"
top18_path = PROJECT_ROOT / "outputs_step3" / "top18_variable_features.csv"

if not model_path.exists():
    raise FileNotFoundError(f"Best EUI model not found: {model_path}")
if not top18_path.exists():
    raise FileNotFoundError(f"Top-18 feature list not found: {top18_path}")

eui_model = joblib.load(model_path)
top18_for_case = pd.read_csv(top18_path).iloc[:, 0].tolist()

case_df = df.copy()
for col in ["orientation_deg", "eui_kwh_m2", "OCEI_kgco2e_m2"]:
    if col not in case_df.columns:
        raise KeyError(f"Required use-case column is missing: {col}")

median_idx = case_df["eui_kwh_m2"].sub(case_df["eui_kwh_m2"].median()).abs().idxmin()
case = case_df.loc[median_idx].copy()
missing_case_features = [c for c in top18_for_case if c not in case_df.columns]
if missing_case_features:
    raise KeyError(f"Use-case top-18 features missing from Step 4 dataframe: {missing_case_features}")

X_case = pd.DataFrame([case[top18_for_case].values], columns=top18_for_case)
eui_pred = float(eui_model.predict(X_case)[0])
eui_sim = float(case["eui_kwh_m2"])
ocei_value = float(case["OCEI_kgco2e_m2"])
abs_error = abs(eui_pred - eui_sim)
pct_error = abs_error / eui_sim * 100 if eui_sim else np.nan

use_case = pd.DataFrame([{
    "sample_id": case.get("sample_id", median_idx),
    "simulated_EUI_kWh_m2_year": eui_sim,
    "predicted_EUI_kWh_m2_year": eui_pred,
    "absolute_error_kWh_m2_year": abs_error,
    "percent_error": pct_error,
    "OCEI_kgCO2e_m2_year": ocei_value,
    "floor_num": case.get("floor_num", np.nan),
    "room_count": case.get("room_count", np.nan),
    "dhw_per_person": case.get("dhw_per_person", np.nan),
    "dhw_temp": case.get("dhw_temp", np.nan),
    "wwr": case.get("wwr", np.nan),
    "cop_cooling": case.get("cop_cooling", np.nan),
    "boiler_eff": case.get("boiler_eff", np.nan),
}])
use_case.to_csv(OUT_DIR / "quantitative_use_case.csv", index=False, encoding="utf-8-sig")

print("=" * 72)
print("QUANTITATIVE USE-CASE: MEDIAN-EUI HOTEL DESIGN")
print("=" * 72)
print(f"Sample ID: {use_case.loc[0, 'sample_id']}")
print(f"Simulated EUI: {eui_sim:.2f} kWh/(m2.year)")
print(f"Predicted EUI: {eui_pred:.2f} kWh/(m2.year)")
print(f"Absolute error: {abs_error:.2f} kWh/(m2.year) ({pct_error:.2f}%)")
print(f"OCEI: {ocei_value:.2f} kgCO2e/(m2.year)")
print("Use-case table saved:", OUT_DIR / "quantitative_use_case.csv")
display(use_case)